# Лабораторная работа 1. Инструменты, данные и первый ориентир

**Курс «Машинное обучение», 4 курс, каф. ФН1**

| | |
|---|---|
| Место в курсе | первое занятие курса, **до** лекции 1 |
| Опора на лекции | нет — работа опирается только на линейную алгебру и матанализ |
| Трудоёмкость | 2 ч аудиторно (части 1–4) + 4 ч самостоятельно |

## Цель работы

Научиться приводить сырую таблицу к виду, пригодному для обучения модели: определять типы признаков, находить и исправлять дефекты данных, разделять выборку на обучающую и контрольную **до** любой обработки и строить ориентир (baseline), с которым дальше сравниваются все модели курса.

Отдельная цель — освоить векторизованные вычисления в NumPy: весь курс написан в матричной записи, и все реализации «с нуля» в работах 2–9 опираются на приёмы из части 1.

## Что нужно сдать

Заполненный ноутбук `lab01_student.ipynb`, в котором:

1. выполнены все задания (ячейки с `# TODO`), код запускается сверху вниз без ошибок;
2. под каждым заданием заполнены ячейки **Вывод** — своими словами, не пересказ кода;
3. в конце — раздел «Итоги работы» с ответами на контрольные вопросы;
4. все графики подписаны (заголовок, оси, легенда).

> **Индивидуальный вариант.** Датасет и набор методов выдаются по вашему ФИО
> (см. ячейку ниже). Отчёт с чужим вариантом не принимается.

## Как устроен практикум

Практикум состоит из 9 работ и **чередуется** с лекциями:

```
Лаб 1 → Лек 1 → Лаб 2 → Лек 2 → … → Лек 8 → Лаб 9
```

Поэтому каждая лабораторная опирается на **уже прочитанные** лекции: работа $k+1$
закрепляет лекцию $k$. Текущая работа идёт первой, до лекции 1, и никакой теории
курса ещё не требует — это подготовка инструментов и данных.

Правила оформления:

* ноутбук должен исполняться сверху вниз в свежем ядре (Kernel → Restart & Run All);
* каждое число, попавшее в вывод, должно быть воспроизводимым — фиксируйте `random_state`;
* графики без подписей осей не засчитываются;
* ячейка **Вывод** — обязательная часть задания, а не украшение.

In [ ]:
# Служебная ячейка: импорты, стиль графиков, воспроизводимость.
import sys, pathlib, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# Модули практикума (variants.py, labdata.py) ищем рядом с ноутбуком.
# Если их нет -- значит, ноутбук открыт в Colab: скачиваем из репозитория курса.
COURSE_FILES_URL = "https://raw.githubusercontent.com/sharipovaka/mltest1/main/notebooks"


def _course_modules_dir():
    here = pathlib.Path.cwd()
    for parent in [here, *here.parents][:4]:
        if (parent / "variants.py").exists():
            return str(parent)
    import urllib.request
    for name in ("variants.py", "labdata.py"):
        if not pathlib.Path(name).exists():
            urllib.request.urlretrieve(f"{COURSE_FILES_URL}/{name}", name)
            print(f"загружен {name} из репозитория курса")
    return str(here)


sys.path.insert(0, _course_modules_dir())
from variants import get_variant, describe_variant  # noqa: E402

RANDOM_STATE = 42          # единый seed на всю работу: результаты воспроизводимы
rng = np.random.default_rng(RANDOM_STATE)

plt.rcParams.update({
    "figure.figsize": (7.5, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("numpy", np.__version__, "| pandas", pd.__version__)

from variants import make_table

## Индивидуальный вариант

Впишите своё ФИО (или почту) в переменную `STUDENT` — вариант вычисляется детерминированно, при повторном запуске он тот же самый.

In [ ]:
STUDENT = "Фамилия Имя Отчество"   # <-- впишите себя

variant = get_variant(STUDENT, lab=1)
describe_variant(variant)

---
# Часть 1. NumPy: векторизация вместо циклов

Весь курс записан в матричных обозначениях. Матрица «объекты–признаки»

$$
F \;=\; \|f_j(x_i)\|_{\ell \times n} \;=\;
\begin{pmatrix}
f_1(x_1) & \dots & f_n(x_1)\\
\vdots & \ddots & \vdots\\
f_1(x_\ell) & \dots & f_n(x_\ell)
\end{pmatrix},
$$

где $\ell$ — число объектов, $n$ — число признаков. В NumPy это массив формы
`(l, n)`. Правило, которое сэкономит вам десятки часов за курс: **если вы пишете
цикл `for` по объектам выборки — почти наверняка есть векторизованная запись,
которая короче и в десятки раз быстрее.**

### Задание 1.1. Попарные расстояния

Даны две матрицы $A \in \mathbb{R}^{m \times n}$ и $B \in \mathbb{R}^{k \times n}$
(строки — объекты). Требуется матрица квадратов евклидовых расстояний
$D \in \mathbb{R}^{m \times k}$, $D_{ij} = \|a_i - b_j\|^2$.

Реализуйте два способа и сравните их по времени и по результату:

1. двойной цикл по $i$ и $j$;
2. векторизованно, по тождеству
$\|a - b\|^2 = \|a\|^2 - 2\langle a, b\rangle + \|b\|^2$.

Эта функция понадобится в работе 7 (метрические методы), поэтому напишите её аккуратно.

In [ ]:
import time


def dists_loops(A, B):
    """Попарные квадраты расстояний двойным циклом (эталон корректности)."""
    # TODO: реализуйте через два вложенных цикла
    raise NotImplementedError


def dists_vectorized(A, B):
    """То же самое без циклов: ||a||^2 - 2<a,b> + ||b||^2."""
    # TODO: реализуйте без единого цикла (подсказка: broadcasting и A @ B.T)
    raise NotImplementedError


A = rng.normal(size=(400, 20))
B = rng.normal(size=(300, 20))

t0 = time.perf_counter(); D_loop = dists_loops(A, B); t_loop = time.perf_counter() - t0
t0 = time.perf_counter(); D_vec = dists_vectorized(A, B); t_vec = time.perf_counter() - t0

print(f"максимальное расхождение: {np.abs(D_loop - D_vec).max():.2e}")
print(f"цикл          : {t_loop*1000:8.1f} мс")
print(f"векторизация  : {t_vec*1000:8.1f} мс")
print(f"ускорение     : {t_loop/t_vec:8.1f} раз")

> **Вывод.** Во сколько раз векторизация быстрее? Почему расхождение не равно строго нулю и почему в векторизованной версии стоит `np.maximum(D, 0)`?
>
> *(ваш ответ здесь)*

### Задание 1.2. Broadcasting: стандартизация вручную

Стандартизация признака — вычитание среднего и деление на стандартное отклонение:

$$
\tilde f_j(x_i) = \frac{f_j(x_i) - \overline{f_j}}{\sigma_j},
\qquad
\overline{f_j} = \frac1\ell \sum_{i=1}^{\ell} f_j(x_i),
\qquad
\sigma_j^2 = \frac1\ell \sum_{i=1}^{\ell} \bigl(f_j(x_i) - \overline{f_j}\bigr)^2 .
$$

Реализуйте её одной строкой на broadcasting'е (без циклов) и проверьте, что у
результата нулевые средние и единичные дисперсии по столбцам.

In [ ]:
X = rng.normal(loc=[10.0, -3.0, 100.0], scale=[2.0, 0.5, 25.0], size=(500, 3))

# TODO: вычислите mean и std по столбцам и стандартизуйте X без циклов
mean = ...
std = ...
X_std = ...

print("средние до :", np.round(mean, 3))
print("средние после:", np.round(X_std.mean(axis=0), 12))
print("ст. откл. после:", np.round(X_std.std(axis=0), 12))

assert np.allclose(X_std.mean(axis=0), 0, atol=1e-10)
assert np.allclose(X_std.std(axis=0), 1, atol=1e-10)
print("OK")

---
# Часть 2. Индивидуальная выборка: первичный осмотр

Таблица генерируется по вашему варианту. Она сознательно «грязная» — примерно
такой вид имеют данные, которые приходят из реальной информационной системы.

Ваша задача в частях 2–7 — превратить её в матрицу «объекты–признаки», пригодную
для обучения, и **честно** оценить, сколько информации в ней есть.

In [ ]:
df_raw, meta = make_table(variant)
TASK = meta["task"]          # 'regression' или 'classification'
TARGET = meta["target"]      # имя целевого столбца

print(f"предметная область : {meta['domain']}")
print(f"тип задачи         : {TASK}")
print(f"целевая переменная : {TARGET}")
print(f"размер таблицы     : {df_raw.shape[0]} объектов x {df_raw.shape[1] - 1} признаков")
df_raw.head(8)

### Задание 2.1. Что вообще лежит в таблице

Выведите: типы столбцов (`dtypes`), число уникальных значений в каждом столбце,
описательные статистики (`describe`) отдельно для числовых и для нечисловых столбцов.

Обратите внимание на столбцы, тип которых `object`, хотя по смыслу они числовые, —
это первое, что придётся чинить.

In [ ]:
# TODO: соберите сводную таблицу: тип, число уникальных значений, число и доля
# пропусков, пример значения — по каждому столбцу.
info = ...
display(info)

# TODO: describe() отдельно для числовых и для object-столбцов

> **Вывод.** Какие столбцы прочитаны как `object`, хотя по смыслу числовые? Какие столбцы выглядят бесполезными уже сейчас?
>
> *(ваш ответ здесь)*

---
# Часть 3. Типы признаков и приведение типов

Признак — это отображение $f: X \to D_f$, и от множества $D_f$ зависит,
что с признаком вообще можно делать:

| Тип | $D_f$ | Пример | Что осмысленно |
|---|---|---|---|
| бинарный | $\{0, 1\}$ | «есть балкон» | всё |
| номинальный | конечное, без порядка | «район» | сравнение на равенство |
| порядковый | конечное, упорядоченное | «ремонт: без / косметический / евро» | сравнение $\le$ |
| количественный | $\mathbb{R}$ | «площадь» | арифметика |

Ошибка кодирования здесь стоит дорого: если номинальный признак «район»
закодировать числами 1..5, модель решит, что «Юг» $>$ «Север» и что
«Запад» ровно посередине между ними.

### Задание 3.1. Починить типы

Напишите функцию `to_numeric_safe(s)`, которая пытается превратить столбец-строку
в числовой: убирает пробелы-разделители разрядов, заменяет запятую на точку и
приводит к `float`. Если после этого больше 20 % значений не распарсились —
столбец действительно нечисловой, возвращаем его как есть.

Примените её ко всем `object`-столбцам.

In [ ]:
def to_numeric_safe(s: pd.Series, threshold: float = 0.2) -> pd.Series:
    """Строковый столбец -> числовой, если он на самом деле числовой."""
    # TODO: снять пробелы-разделители, заменить ',' на '.', привести к числу.
    # Если доля непреобразованных непустых значений > threshold — вернуть s без изменений.
    raise NotImplementedError


df = df_raw.copy()
for col in df.columns:
    before = df[col].dtype
    df[col] = to_numeric_safe(df[col])
    if before != df[col].dtype:
        print(f"{col:28s}: {before} -> {df[col].dtype}")

print("\nТипы после приведения:")
print(df.dtypes.value_counts())

### Задание 3.2. Нормализация категорий

В категориальных столбцах одно и то же значение встречается в разных написаниях
(`'Москва'`, `'москва '`, `'МОСКВА'`). Для компьютера это три разные категории.

Приведите категориальные столбцы к каноническому виду и сравните число уникальных
значений до и после.

In [ ]:
cat_cols = meta["categorical"]

print("до нормализации:")
for col in cat_cols:
    print(f"  {col:22s}: {df[col].nunique()} значений -> {sorted(df[col].dropna().unique())[:6]}")

# TODO: привести строковые значения к нижнему регистру и убрать краевые пробелы.
# Осторожно: пропуски (NaN) не являются строками.

print("\nпосле нормализации:")
for col in cat_cols:
    print(f"  {col:22s}: {df[col].nunique()} значений -> {sorted(df[col].dropna().unique())}")

> **Вывод.** Во сколько раз сократилось число категорий? Что случилось бы с One-Hot кодированием, если бы вы этот шаг пропустили?
>
> *(ваш ответ здесь)*

---
# Часть 4. Дефекты данных

Разберём четыре типовых дефекта: дубликаты, вырожденные признаки, пропуски и выбросы.

### Задание 4.1. Дубликаты и вырожденные признаки

Найдите полные дубликаты строк и удалите их. Найдите признаки с одним уникальным
значением (нулевая дисперсия) и признаки, у которых доля самого частого значения
превышает 99 %, — они не несут информации о различиях между объектами.

In [ ]:
# TODO: посчитайте и удалите полные дубликаты строк
n_dup = ...
print(f"полных дубликатов строк: {n_dup} ({n_dup / len(df) * 100:.1f} %)")

# TODO: найдите константные и почти константные (доля моды > 0.99) признаки,
# не трогая целевую переменную, и удалите их
const_cols, quasi_const_cols = [], []

### Задание 4.2. Пропуски и их механизм

Мало посчитать долю пропусков — важно понять, **почему** значение отсутствует.
Различают:

* **MCAR** (missing completely at random) — пропуск не зависит ни от чего;
* **MAR** (missing at random) — вероятность пропуска зависит от *других наблюдаемых*
  признаков (например, доход чаще не указывают клиенты с малым стажем);
* **MNAR** — вероятность пропуска зависит от самого пропущенного значения
  (не указывают именно очень большие доходы). Худший случай: по данным его не отличить.

Постройте столбчатую диаграмму долей пропусков. Затем для каждого столбца с
пропусками проверьте гипотезу MAR: сравните средние значения **остальных**
числовых признаков в группах «значение есть» и «значение пропущено».
Если различия заметны — механизм не MCAR, и заполнять пропуск средним по всей
выборке опасно.

In [ ]:
# TODO: 1) горизонтальная столбчатая диаграмма долей пропусков по признакам
# TODO: 2) для каждого столбца с пропусками — сравнение средних остальных
#          числовых признаков в группах «есть значение» / «пропущено».
#          Разницу удобно мерить в единицах стандартного отклонения признака.

> **Вывод.** Для каких столбцов механизм пропусков похож на MCAR, а для каких — на MAR? Как это меняет способ заполнения?
>
> *(ваш ответ здесь)*

### Задание 4.3. Выбросы

Два рабочих правила:

* **правило $3\sigma$**: $|x - \overline{x}| > 3\sigma$ — годится для примерно
  нормальных признаков и само страдает от выбросов (они раздувают $\sigma$);
* **правило межквартильного размаха**: $x < Q_1 - 1{.}5\,\mathrm{IQR}$ или
  $x > Q_3 + 1{.}5\,\mathrm{IQR}$, где $\mathrm{IQR} = Q_3 - Q_1$ —
  устойчиво к самим выбросам.

Постройте `boxplot` для числовых признаков (в стандартизованных единицах, иначе
всё сольётся), посчитайте число выбросов по обоим правилам и **посмотрите на сами
объекты-выбросы**: часть из них — опечатки (значение больше правдоподобного в 100 раз),
часть — настоящие редкие объекты.

In [ ]:
num_cols = df.select_dtypes(include="number").columns.drop(TARGET)

# TODO: 1) boxplot по стандартизованным числовым признакам
# TODO: 2) таблица: число выбросов по правилу IQR и по правилу 3 sigma
# TODO: 3) вывести сами объекты с самыми большими значениями подозрительного признака

### Задание 4.4. Исправление опечаток масштаба

Среди выбросов есть особая группа: значения, превышающие 99-й перцентиль примерно
в 100 раз. Это не редкие объекты, а **ошибка ввода** — потерянный десятичный
разделитель (год `2010` записан как `201000`, пробег `85 000` как `8 500 000`).

Найдите такие значения по правилу «больше 30 перцентилей $q_{0.99}$», разделите
их на 100 и покажите, как изменилась корреляция признака с целевой переменной.
Настоящие выбросы (превышение в 4–12 раз) не трогайте: они правдоподобны.

In [ ]:
# TODO: для каждого числового признака найдите значения, превышающие 30 * q_{0.99},
#       разделите их на 100 и сравните корреляцию признака с целью до и после.

> **Вывод.** Какое правило нашло больше выбросов и почему? Какие из найденных объектов — опечатки, а какие — настоящие редкие наблюдения? Что вы с ними сделаете?
>
> *(ваш ответ здесь)*

---
# Часть 5. Визуальный анализ

Три графика, которые стоит строить всегда: распределение каждого признака,
распределение цели в разрезе категорий и матрица корреляций.

In [ ]:
num_cols = df.select_dtypes(include="number").columns.drop(TARGET)
cat_cols = [c for c in meta["categorical"] if c in df.columns]

# TODO: (1) сетка гистограмм по числовым признакам
# TODO: (2) распределение целевой переменной в разрезе каждого категориального
#           признака (boxplot для регрессии, доля класса 1 для классификации)
# TODO: (3) тепловая карта матрицы корреляций числовых признаков вместе с целью,
#           и топ-5 признаков по |корреляции| с целью

> **Вывод.** Какие признаки сильнее всего связаны с целью? Есть ли пары сильно коррелированных между собой признаков? Чем это грозит?
>
> *(ваш ответ здесь)*

---
# Часть 6. Ловушка: признак, которого не должно быть

В таблице есть столбец `id`. Формально это число, и его можно подать в модель.
Проверьте, что будет.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

# TODO: обучите решающее дерево ГЛУБИНЫ 4 на одном-единственном признаке id
#       и оцените качество по 5-блочной кросс-валидации
#       (r2 для регрессии, roc_auc для классификации).
# TODO: постройте диаграмму рассеяния "id против цели".
# TODO: удалите столбец id из df.

> **Вывод.** Откуда взялась такая связь и почему признак `id` необходимо удалить, хотя он улучшает качество?
>
> *(ваш ответ здесь)*

---
# Часть 7. Разбиение выборки и предобработка без утечек

Ключевое правило всего курса:

> Все параметры предобработки (средние, дисперсии, медианы для заполнения
> пропусков, список категорий) вычисляются **только по обучающей части**
> и затем применяются к контрольной.

Иначе информация о контрольной выборке просачивается в обучение, и оценка
качества оказывается завышенной. Строгое обоснование — лекция 4 и работа 5,
где мы измерим величину этого завышения.

Технически это удобно делать через `Pipeline` и `ColumnTransformer`: они хранят
параметры, подобранные на `fit`, и применяют их в `transform`.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X = df.drop(columns=[TARGET])
y = df[TARGET]

# TODO: 1) разделите столбцы на числовые и категориальные
# TODO: 2) train_test_split, test_size=0.25, random_state=RANDOM_STATE,
#          для классификации — со стратификацией по y
# TODO: 3) соберите ColumnTransformer:
#          числовые   -> SimpleImputer(median) + StandardScaler
#          категории  -> SimpleImputer(most_frequent) + OneHotEncoder(handle_unknown="ignore")
# TODO: 4) fit_transform на обучающей, transform на контрольной

### Задание 7.1. Цена нарушения правила

Проверьте эмпирически, что порядок операций важен: вычислите среднее и стандартное
отклонение признака (а) по всей выборке и (б) только по обучающей части.
Насколько отличаются полученные стандартизованные значения на контроле?

Это «мягкая» утечка — она почти не портит качество, но именно она даёт первое
интуитивное понимание, зачем нужен `Pipeline`. Количественную оценку ущерба
от жёстких утечек мы получим в работе 5.

In [ ]:
# TODO: сравните стандартизацию контрольной части по статистикам всей выборки
#       и по статистикам только обучающей части

---
# Часть 8. Ориентир: с чем сравнивать модели

Прежде чем радоваться качеству модели, нужно знать, сколько даёт **тривиальный
ответ**: константа. Для регрессии это среднее (оптимум для квадратичной ошибки)
или медиана (оптимум для абсолютной); для классификации — самый частый класс.

Мера ошибки на этом этапе вводится «по определению» — строгое понятие функции
потерь и эмпирического риска появится на лекции 1:

$$
\mathrm{MSE} = \frac1{\ell}\sum_{i=1}^{\ell} (a(x_i) - y_i)^2, \qquad
\mathrm{MAE} = \frac1{\ell}\sum_{i=1}^{\ell} |a(x_i) - y_i|, \qquad
R^2 = 1 - \frac{\sum_i (a(x_i) - y_i)^2}{\sum_i (\overline{y} - y_i)^2}.
$$

Заметьте: $R^2 = 0$ — это ровно качество константного прогноза средним.

In [ ]:
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (accuracy_score, mean_absolute_error,
                             mean_squared_error, r2_score, roc_auc_score)

# TODO: обучите на X_train_t два алгоритма и сравните их на контроле:
#   регрессия     : DummyRegressor(strategy='mean')  и LinearRegression
#                   метрики RMSE, MAE, R2
#   классификация : DummyClassifier(strategy='most_frequent') и LogisticRegression
#                   метрики accuracy, ROC-AUC
# Результат оформите таблицей.

> **Вывод.** Насколько модель лучше константы? Что означает `accuracy = 0.8` у константного классификатора, если классы несбалансированы?
>
> *(ваш ответ здесь)*

---
# Часть 9. Сохранение результата

Подготовленные данные понадобятся в следующих работах. Сохраните:

* матрицы `X_train_t`, `X_test_t` и векторы `y_train`, `y_test` — в `data/prepared.npz`;
* очищенную таблицу до кодирования — в `data/clean.csv`;
* сам объект `preprocessor` — через `joblib`, чтобы применить его к новым данным.

In [ ]:
import pathlib
import joblib

data_dir = pathlib.Path("data")
data_dir.mkdir(exist_ok=True)

np.savez_compressed(
    data_dir / "prepared.npz",
    X_train=X_train_t, X_test=X_test_t,
    y_train=np.asarray(y_train), y_test=np.asarray(y_test),
    feature_names=np.asarray(feature_names, dtype=object),
)
df.to_csv(data_dir / "clean.csv", index=False, encoding="utf-8")
joblib.dump(preprocessor, data_dir / "preprocessor.joblib")

print("сохранено:", *[p.name for p in sorted(data_dir.iterdir())])

## Итоги работы

Ответьте письменно на контрольные вопросы (по 2–4 предложения):

1. Чем номинальный признак отличается от порядкового и почему их нельзя кодировать одинаково? Приведите пример из своей таблицы.
2. Что такое утечка данных? Приведите два примера: один из вашей таблицы, второй — придуманный для задачи «предсказать, вернёт ли клиент кредит».
3. Почему параметры масштабирования нужно оценивать только по обучающей выборке? Что именно завышается, если это правило нарушить?
4. Правило $3\sigma$ нашло меньше выбросов, чем правило IQR. Объясните, почему так вышло, через определения обеих величин.
5. Вы получили accuracy 0.93. Какие ещё два числа нужно знать, чтобы понять, хороший это результат или нет?

### Домашнее задание

1. **Заполнение пропусков с учётом механизма.** Для столбца, который вы определили как MAR, реализуйте два способа заполнения: (а) общей медианой; (б) медианой внутри групп, заданных квартилями признака-драйвера. Сравните гистограммы заполненного признака с гистограммой на объектах без пропуска. Какой способ меньше искажает распределение? Ответ подкрепите числом — расстоянием Колмогорова–Смирнова (`scipy.stats.ks_2samp`).

2. **Своя реализация OneHotEncoder.** Напишите функцию `one_hot(series, categories=None)`, которая возвращает матрицу индикаторов и список категорий; при `categories`, заданном извне, неизвестные значения кодируются нулевой строкой. Проверьте совпадение со `sklearn.preprocessing.OneHotEncoder(handle_unknown='ignore')` на вашем категориальном признаке. Объясните, зачем нужен параметр `drop='first'` и в каком случае его отсутствие ломает линейную регрессию (подсказка: сумма индикаторов равна столбцу единиц).